# 01 - Data Preprocessing

Este notebook realiza la carga, selección, limpieza y transformación inicial de los datos de la Encuesta de Cultura Política. 

El objetivo es construir un dataset limpio para la predicción de orientación política, usando la variable P5328 como variable objetivo.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

### 1. Definición de rutas

In [ ]:
RAW_DATA_PATH = Path("../data/raw/empalme_politica.csv")
PROCESSED_DATA_PATH = Path("../data/processed/political_orientation_clean.csv")

PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

### 2. Carga de datos

In [ ]:
data = pd.read_csv(RAW_DATA_PATH)

print("Dimensiones iniciales del dataset:", data.shape)
data.head()

### 3. Selección de variables

In [ ]:
columnas = [
    "P220",
    "P2057",
    "P605",
    "P6210",
    "P6945",
    "P3039",
    "P2001S1",
    "P2001S2",
    "P2001S3",
    "P2001S4",
    "P2001S17",
    "P2001S7",
    "P2001S12",
    "P2001S13",
    "P2001S14",
    "P2001S15",
    "P2001S18",
    "P2003S4",
    "P2001S10",
    "P2003S17",
    "P2003S7",
    "P2003S10",
    "P2003S12",
    "P2003S13",
    "P2003S14",
    "P2003S15",
    "P2003S18",
    "P5373S4",
    "P5373S5",
    "P5373S6",
    "P5306S1",
    "P5306S2",
    "P2019",
    "P2016S1",
    "P2016S2",
    "P2016S3",
    "P2016S4",
    "P2016S5",
    "P2016S6",
    "P2016S7",
    "P2016S8",
    "P2016S9",
    "P2017S1",
    "P2017S2",
    "P2017S3",
    "P2017S4",
    "P2017S5",
    "P2017S6",
    "P2021",
    "P1754",
    "P5376S1",
    "P5376S2",
    "P5376S3",
    "P5376S4",
    "P5376S5",
    "P5376S6",
    "P5317S4",
    "P5317S5",
    "P5307S1",
    "P5307S2",
    "P5307S3",
    "P5307S4",
    "P5307S5",
    "P5328"
]

In [ ]:
empalme_politica = data[columnas].copy()

print("Dimensiones después de seleccionar variables:", empalme_politica.shape)
empalme_politica.head()

### 4. Tratamiento valores faltantes

In [ ]:
missing_values = empalme_politica.isna().sum()
missing_values[missing_values > 0].sort_values(ascending=False)

In [ ]:
initial_rows = len(empalme_politica)

empalme_politica = empalme_politica.dropna()

final_rows = len(empalme_politica)
removed_rows = initial_rows - final_rows
removed_percentage = (removed_rows / initial_rows) * 100

print("Filas iniciales:", initial_rows)
print("Filas después de eliminar NaN:", final_rows)
print(f"Porcentaje eliminado: {removed_percentage:.2f}%")

### 5. Revisión variable objetivo P5328

In [ ]:
empalme_politica["P5328"].value_counts().sort_index()

### 6. Tratamiento respuestas no válidas

Respuestas tipo “no informa” o “no sabe”.

In [ ]:
empalme_politica = empalme_politica[
    ~empalme_politica["P5328"].isin([98, 99])
]

print("Dimensiones después de eliminar 98 y 99:", empalme_politica.shape)
empalme_politica["P5328"].value_counts().sort_index()

### 7. Tratamiento categorías centrales

El análisis será binario izquierda/derecha, se excluyen valores centrales.

In [ ]:
empalme_politica = empalme_politica[
    ~empalme_politica["P5328"].isin([5, 6])
]

print("Dimensiones después de eliminar categorías centrales:", empalme_politica.shape)
empalme_politica["P5328"].value_counts().sort_index()

### 8. Transformación variable objetivo

In [ ]:
empalme_politica["political_orientation"] = np.where(
    empalme_politica["P5328"] <= 4,
    0,
    1
)

In [ ]:
empalme_politica = empalme_politica.drop(columns=["P5328"])

In [ ]:
#Distribución final del target

empalme_politica["political_orientation"].value_counts(normalize=True) * 100

### 9. Guardar dataset limpio

In [ ]:
empalme_politica.to_csv(PROCESSED_DATA_PATH, index=False)

print("Dataset limpio guardado en:", PROCESSED_DATA_PATH)
print("Dimensiones finales:", empalme_politica.shape)

In [ ]:
clean_data = pd.read_csv(PROCESSED_DATA_PATH)

clean_data.head()

In [ ]:
clean_data.info()